# KSL Few-Shot Study — Table 5 (all 6 models)

Runs K = 1, 3, 10 shot fine-tuning of the KSL head on top of each model's
**best Stage-1 encoder from Table 2**, for all six rows of Table 5: RNN,
LSTM, BiLSTM, GRU, BiGRU, and Mask-Aware GCN + Transformer (ours).

`build_baseline()` / `MultiHeadRecurrentModel` are deliberately drop-in
replacements for `MultiHeadSignModel` (same `.encoder` / `.heads` /
`forward(feats, vis, fmask, head=)` shape -- see `baselines.py`'s
docstring), so one training loop below covers all six rows; only
`build_model()` branches on which architecture to construct.

Protocol (same as before, now per model):
1. Split KSL data once into a **fixed eval set** (shared by every model, K,
   and trial) and a **pool** every K-shot support set is drawn from.
2. For each (model, K), draw the support set, fine-tune a fresh copy of that
   model's pretrained encoder + a new Khmer head on just the support set,
   evaluate on the fixed eval set.
3. Repeat each (model, K) over `N_TRIALS` different random supports and
   report mean ± std.

**Set `PRETRAINED` below for every model** to whichever checkpoint is that
architecture's "best Stage-1 encoder" per your Table 2 -- a baseline's
pretrained checkpoint comes from
`train_multilingual.py --model <rnn|lstm|bilstm|gru|bigru>` (see that
script's docstring). **Any model whose checkpoint file doesn't exist yet is
skipped automatically** (printed, not a crash) so you can fill in Table 5
incrementally as more baselines finish pretraining.


## 1. Imports

In [1]:
import sys, json, time
from pathlib import Path

_here = (Path("__file__").resolve().parent
         if "__file__" in dir() else Path(".").resolve())
_root = _here.parent
sys.path.insert(0, str(_root / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm

from ksl_recognition.data.augmentations import AugmentPipeline
from ksl_recognition.data.dataset import (KSLLandmarkDataset, PadCollate,
                                           scan_landmarks, stratified_split, k_shot_sample)
from ksl_recognition.models.sign_model import MultiHeadSignModel, count_parameters
from ksl_recognition.models.baselines import build_baseline, BASELINE_PRESETS

print("torch", torch.__version__)


torch 2.12.0


## 2. Configuration

In [ ]:
DATA_DIR  = "../data/processed_landmarks/ksl"
EVAL_FRAC = 0.3     # held out ONCE, shared across every model / K / trial
SEED      = 67

K_VALUES  = [1, 3, 10]
N_TRIALS  = 5        # independent random supports per (model, K) -- mean +/- std
FREEZE_ENCODER = False   # False = fine-tune encoder + head (paper's stated
                          # preference: unfrozen transfers better than freeze)

# ── Few-shot fine-tuning hyperparameters -- gentler than full-supervised
# training (tiny data): fewer epochs would underfit, heavy augmentation would
# distort the few examples you have, but light augmentation still helps
# avoid instantly memorizing 1-10 examples per class.
EPOCHS_FEWSHOT   = 60
ENC_LR_FEWSHOT   = 5e-5
HEAD_LR_FEWSHOT  = 5e-4
WEIGHT_DECAY     = 1e-4
BATCH_SIZE       = 128
MAX_LEN          = 128
WORKERS          = 2

IN_CH = 5   # x, y, dx, dy, vis

# ── "ours" architecture -- MUST match its PRETRAINED checkpoint exactly ────
GCN_ARCH = dict(d_model=256, gcn_hidden=(128, 256), nhead=8, num_layers=6,
                 dim_ff=512, dropout=0.25, max_len=MAX_LEN)

# ── baseline architecture -- MUST match train_baselines.py's defaults used
# when that checkpoint was pretrained (hidden=256, num_layers=2, dropout=0.3)
BASELINE_ARCH = dict(hidden=256, num_layers=2, dropout=0.3)

# One "best Stage-1 encoder from Table 2" checkpoint per model. Baseline
# checkpoints come from:
#   python -m ksl_recognition.training.train_multilingual --model <name> \
#       --languages <winning combo per Table 2> --out checkpoints/multilingual_pretrain_<name>.pt
PRETRAINED = {
    "ours":   "../top-3/multilingual_pretrain.pt",   # your arabic+argentina+rsl Stage-1 run
    "rnn":    "../best_Pretrain_model_RNN/multilingual_pretrain_rnn.pt",   # all-7-language winner
    "lstm":   "../best_Pretrain_model_RNN/multilingual_pretrain_lstm.pt",
    "bilstm": "../best_Pretrain_model_RNN/multilingual_pretrain_bilstm.pt",
    "gru":    "../best_Pretrain_model_RNN/multilingual_pretrain_gru.pt",
    "bigru":  "../best_Pretrain_model_RNN/multilingual_pretrain_bigru.pt",
}
MODEL_LABELS = {
    "rnn": "RNN", "lstm": "LSTM", "bilstm": "BiLSTM", "gru": "GRU", "bigru": "BiGRU",
    "ours": "Mask-Aware GCN + Transformer (ours)",
}
MODELS = list(PRETRAINED)   # run order = Table 5 row order

RESULTS_PATH = Path("../plots/fewshot_results.json")
Path("../plots").mkdir(parents=True, exist_ok=True)

def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = pick_device()
print("device:", device)
print("models:", MODELS)
print("K values:", K_VALUES, "| trials per (model, K):", N_TRIALS, "| freeze_encoder:", FREEZE_ENCODER)
for name, path in PRETRAINED.items():
    print(f"  {name:8s} {'OK  ' if Path(path).exists() else 'MISSING'}  {path}")


## 3. Data: one fixed eval set, one pool

Scanned/split ONCE and reused for every model, K, and trial below -- that's
what makes every number in the eventual table comparable to every other.

In [3]:
items, class_to_idx = scan_landmarks(DATA_DIR)
num_classes = len(class_to_idx)
pool_items, eval_items = stratified_split(items, EVAL_FRAC, SEED)
print(f"{len(items)} clips | {num_classes} classes | "
      f"pool {len(pool_items)} (support sampled from here) / eval {len(eval_items)} (fixed)")


scan_landmarks: skipped 234 clip(s) under 3 frames
4818 clips | 194 classes | pool 3384 (support sampled from here) / eval 1434 (fixed)


## 4. Builders

In [4]:
def build_eval_loader():
    ds = KSLLandmarkDataset(eval_items, class_to_idx, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                       collate_fn=PadCollate(use_velocity=True), num_workers=WORKERS)


def build_support_loader(k, trial_seed):
    support_items = k_shot_sample(pool_items, k, seed=trial_seed)
    augmenter = AugmentPipeline.light()   # gentle -- see config cell rationale
    ds = KSLLandmarkDataset(support_items, class_to_idx, MAX_LEN, augment=augmenter)
    bs = min(BATCH_SIZE, len(support_items))
    return DataLoader(ds, batch_size=bs, shuffle=True,
                       collate_fn=PadCollate(use_velocity=True), num_workers=WORKERS,
                       drop_last=False), len(support_items)


def build_model(model_name):
    """Either MultiHeadSignModel ("ours") or one of the RNN-family baselines
    -- both expose the same .encoder / .heads / forward(...) shape, so
    everything downstream (optimizer, freeze logic, eval) is identical."""
    if model_name == "ours":
        model = MultiHeadSignModel(
            in_ch=IN_CH, head_classes={"ksl": num_classes}, **GCN_ARCH,
        ).to(device)
        for m in model.modules():
            if isinstance(m, nn.TransformerEncoder):
                m.enable_nested_tensor = False
                m.use_nested_tensor = False
    else:
        model = build_baseline(
            model_name, in_ch=IN_CH, head_classes={"ksl": num_classes}, **BASELINE_ARCH,
        ).to(device)

    ckpt = torch.load(PRETRAINED[model_name], map_location=device)
    ckpt_model_type = ckpt.get("model_type")
    if ckpt_model_type and model_name != "ours" and ckpt_model_type != model_name:
        raise ValueError(f"{PRETRAINED[model_name]} was pretrained as {ckpt_model_type!r}, "
                         f"can't warm-start a {model_name!r} encoder with it")
    state = ckpt.get("model", ckpt)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  loaded {PRETRAINED[model_name]}  missing={len(missing)}  unexpected={len(unexpected)}")

    if FREEZE_ENCODER:
        for p in model.encoder.parameters():
            p.requires_grad = False
    return model


def build_optimizer(model):
    params = [{"params": model.heads.parameters(), "lr": HEAD_LR_FEWSHOT}]
    if not FREEZE_ENCODER:
        params.append({"params": model.encoder.parameters(), "lr": ENC_LR_FEWSHOT})
    return torch.optim.AdamW(params, weight_decay=WEIGHT_DECAY)


@torch.no_grad()
def evaluate_topk(model, loader, k_list=(1, 5)):
    model.eval()
    all_logits, all_labels = [], []
    for feats, vis, fmask, labels in loader:
        feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
        logits = model(feats, vis, fmask, head="ksl")
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    out = {}
    for k in k_list:
        if k >= logits.shape[1]:
            continue
        topk = logits.topk(k, dim=1).indices
        out[k] = (topk == labels.unsqueeze(1)).any(1).float().mean().item()
    return out


## 5. One trial: fine-tune on a K-shot support set, evaluate on the fixed eval set

A fresh model (reloaded from that model's `PRETRAINED` checkpoint) every
trial -- no leakage of weights between trials, K values, or models.

In [5]:
def run_trial(model_name, k, trial_seed):
    support_loader, n_support = build_support_loader(k, trial_seed)
    model = build_model(model_name)
    optim = build_optimizer(model)

    model.train()
    for epoch in range(EPOCHS_FEWSHOT):
        for feats, vis, fmask, labels in support_loader:
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
            logits = model(feats, vis, fmask, head="ksl")
            loss = F.cross_entropy(logits, labels)
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

    eval_loader = build_eval_loader()
    topk = evaluate_topk(model, eval_loader, k_list=(1, 5))
    return topk, n_support


resuming -- 8 trial(s) already done

=== ours  K=3  trial 4/5  (seed=3070) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=582  top1=0.6646  top5=0.8591  (839s)

=== ours  K=3  trial 5/5  (seed=3071) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=582  top1=0.6681  top5=0.8682  (833s)

=== ours  K=10  trial 1/5  (seed=10067) ===
k_shot_sample: 31 class(es) had fewer than k=10 items in the pool -- used all available for those
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=1873  top1=0.8410  top5=0.9547  (1178s)

=== ours  K=10  trial 2/5  (seed=10068) ===
k_shot_sample: 31 class(es) had fewer than k=10 items in the pool -- used all available for those
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=1873  top1=0.8417  top5=0.9484  (1189s)

=== ours  K=10  trial 3/5  (seed=10069) ===
k_shot_sample: 31 class(es) had fewer than k=10 items in the pool -- used all available for those
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=1873  top1=0.8403  top5=0.9519  (1149s)

=== ours  K=10  trial 4/5  (seed=10070) ===
k_shot_sample: 31 class(es) had fewer than k=10 items in the pool -- used all available for those
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=1873  top1=0.8291  top5=0.9442  (1153s)

=== ours  K=10  trial 5/5  (seed=10071) ===
k_shot_sample: 31 class(es) had fewer than k=10 items in the pool -- used all available for those
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=1873  top1=0.8445  top5=0.9526  (1159s)

=== rnn  K=1  trial 1/5  (seed=1067) ===
  loaded ../best_Pretrain_model_RNN/multilingual_pretrain_rnn.pt  missing=2  unexpected=14

## 6. Run every model x K x trial

Resumable: results are saved to `RESULTS_PATH` after every single trial, so
an interrupt only costs the trials not yet finished. Any model whose
`PRETRAINED` checkpoint is missing is skipped (with a printed reason) rather
than aborting the whole run.

In [7]:
FORCE_RERUN = False

results = {}
if RESULTS_PATH.exists() and not FORCE_RERUN:
    results = json.loads(RESULTS_PATH.read_text())
    done = sum(len(v) for m in results.values() for v in m.values())
    print(f"resuming -- {done} trial(s) already done")

for model_name in MODELS:
    if not Path(PRETRAINED[model_name]).exists():
        print(f"skipping {model_name}: {PRETRAINED[model_name]} not found")
        continue
    results.setdefault(model_name, {})
    for k in K_VALUES:
        key = str(k)
        results[model_name].setdefault(key, [])
        while len(results[model_name][key]) < N_TRIALS:
            trial_idx = len(results[model_name][key])
            trial_seed = SEED + 1000 * k + trial_idx     # deterministic, distinct per (k, trial)
            print(f"\n=== {model_name}  K={k}  trial {trial_idx + 1}/{N_TRIALS}  (seed={trial_seed}) ===")
            t0 = time.time()
            topk, n_support = run_trial(model_name, k, trial_seed)
            print(f"  n_support={n_support}  top1={topk.get(1):.4f}  top5={topk.get(5):.4f}  "
                  f"({time.time() - t0:.0f}s)")
            results[model_name][key].append({"trial": trial_idx, "seed": trial_seed,
                                              "n_support": n_support, "topk": topk})
            RESULTS_PATH.write_text(json.dumps(results, indent=2))   # save after EVERY trial

print("\nall done." if all(
    len(results.get(m, {}).get(str(k), [])) >= N_TRIALS
    for m in MODELS if Path(PRETRAINED[m]).exists() for k in K_VALUES
) else "\nstopped early, or some models were skipped -- rerun this cell after adding missing checkpoints.")


resuming -- 33 trial(s) already done

=== bigru  K=1  trial 1/1  (seed=1067) ===
  loaded ../best_Pretrain_model_RNN/multilingual_pretrain_bigru.pt  missing=2  unexpected=2
  n_support=194  top1=0.4428  top5=0.6792  (760s)

=== bigru  K=3  trial 1/1  (seed=3067) ===
  loaded ../best_Pretrain_model_RNN/multilingual_pretrain_bigru.pt  missing=2  unexpected=2


KeyboardInterrupt: 

In [ ]:
=== ours  K=1  trial 1/5  (seed=1067) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=194  top1=0.4554  top5=0.6799  (732s)

=== ours  K=1  trial 2/5  (seed=1068) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=194  top1=0.4561  top5=0.6869  (729s)

=== ours  K=1  trial 3/5  (seed=1069) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=194  top1=0.4763  top5=0.7008  (732s)

=== ours  K=1  trial 4/5  (seed=1070) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=194  top1=0.4407  top5=0.6674  (735s)

=== ours  K=1  trial 5/5  (seed=1071) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=194  top1=0.4623  top5=0.6597  (750s)

=== ours  K=3  trial 1/5  (seed=3067) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=582  top1=0.7001  top5=0.8835  (871s)

=== ours  K=3  trial 2/5  (seed=3068) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=582  top1=0.6841  top5=0.8849  (855s)

=== ours  K=3  trial 3/5  (seed=3069) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6
  n_support=582  top1=0.6806  top5=0.8626  (841s)

=== ours  K=3  trial 4/5  (seed=3070) ===
  loaded ../top-3/multilingual_pretrain.pt  missing=2  unexpected=6

## 7. Build Table 5

In [8]:
def _mean_std(vals):
    arr = np.array(vals)
    return arr.mean(), arr.std()

rows = []
for model_name in MODELS:
    row = {"Model": MODEL_LABELS[model_name]}
    model_results = results.get(model_name, {})
    for k in K_VALUES:
        trials = model_results.get(str(k), [])
        if not trials:
            row[f"K = {k}"] = "—"
            continue
        top1s = [t["topk"].get("1", t["topk"].get(1)) for t in trials]
        mean, std = _mean_std(top1s)
        row[f"K = {k}"] = f"{mean * 100:.2f} ± {std * 100:.2f}"
    rows.append(row)

table5 = pd.DataFrame(rows)
table5.to_excel("../plots/fewshot_table5.xlsx", index=False)
table5


,Model,K = 1,K = 3,K = 10
0,Mask-Aware GCN + Transformer (ours),45.82 ± 1.15,67.95 ± 1.27,83.93 ± 0.53
1,RNN,36.51 ± 1.30,56.69 ± 0.15,73.29 ± 0.00
2,LSTM,37.03 ± 0.00,57.03 ± 0.00,79.36 ± 0.00
3,BiLSTM,40.66 ± 0.00,40.66 ± 0.00,40.66 ± 0.00
4,GRU,40.66 ± 0.00,40.66 ± 0.00,40.66 ± 0.00
5,BiGRU,44.28 ± 0.00,—,—


## 8. Paste-ready text for the paper

In [9]:
for model_name in MODELS:
    model_results = results.get(model_name, {})
    if not model_results:
        print(f"{MODEL_LABELS[model_name]}: skipped (no checkpoint)")
        continue
    print(f"{MODEL_LABELS[model_name]}:")
    for k in K_VALUES:
        trials = model_results.get(str(k), [])
        if not trials:
            print(f"  K={k}: (not run)")
            continue
        top1s = [t["topk"].get("1", t["topk"].get(1)) for t in trials]
        top5s = [t["topk"].get("5", t["topk"].get(5)) for t in trials]
        m1, s1 = _mean_std(top1s)
        m5, s5 = _mean_std(top5s)
        print(f"  K={k}: top-1 {m1*100:.2f}% (+/- {s1*100:.2f}), top-5 {m5*100:.2f}% (+/- {s5*100:.2f})  "
              f"over {len(top1s)} trial(s)")


Mask-Aware GCN + Transformer (ours):
  K=1: top-1 45.82% (+/- 1.15), top-5 67.89% (+/- 1.45)  over 5 trial(s)
  K=3: top-1 67.95% (+/- 1.27), top-5 87.17% (+/- 1.07)  over 5 trial(s)
  K=10: top-1 83.93% (+/- 0.53), top-5 95.03% (+/- 0.37)  over 5 trial(s)
RNN:
  K=1: top-1 36.51% (+/- 1.30), top-5 61.66% (+/- 1.26)  over 5 trial(s)
  K=3: top-1 56.69% (+/- 0.15), top-5 81.06% (+/- 1.00)  over 3 trial(s)
  K=10: top-1 73.29% (+/- 0.00), top-5 89.61% (+/- 0.00)  over 1 trial(s)
LSTM:
  K=1: top-1 37.03% (+/- 0.00), top-5 59.97% (+/- 0.00)  over 1 trial(s)
  K=3: top-1 57.03% (+/- 0.00), top-5 59.97% (+/- 0.00)  over 1 trial(s)
  K=10: top-1 79.36% (+/- 0.00), top-5 94.00% (+/- 0.00)  over 1 trial(s)
BiLSTM:
  K=1: top-1 40.66% (+/- 0.00), top-5 62.41% (+/- 0.00)  over 1 trial(s)
  K=3: top-1 40.66% (+/- 0.00), top-5 62.41% (+/- 0.00)  over 1 trial(s)
  K=10: top-1 40.66% (+/- 0.00), top-5 62.41% (+/- 0.00)  over 1 trial(s)
GRU:
  K=1: top-1 40.66% (+/- 0.00), top-5 62.41% (+/- 0.00)  ov